# Thu thập Đánh giá Sản phẩm Tiki (Crawl Product Reviews)

Notebook thu thập đánh giá (`reviews`) từ API Tiki:
1. **Chọn sản phẩm cần crawl**: lấy `review_count` mới nhất của mỗi sản phẩm trong `product_detail.csv`, so với số review đã lưu trong `product_review.csv`. Chỉ crawl sản phẩm có `review_count` > số review đã lưu (lần đầu: tất cả sản phẩm có review).
2. **Ghép Seller**: lấy `seller_product_id` / `seller_id` từ `product_seller.csv` (ưu tiên `is_current_seller = True`), loại giá trị rỗng/`nan`.
3. **Crawl mới nhất trước** (`id|desc`) và **early stop** khi chạm review đã lưu.
4. **An toàn**: chỉ lưu review của sản phẩm đã crawl xong không lỗi; dừng toàn bộ khi bị chặn (401/403); ghi lại danh sách sản phẩm lỗi; luôn lưu buffer trong `finally`.

In [ ]:
import os
import random
import time
from datetime import datetime
from pathlib import Path
import pandas as pd
import requests

REVIEW_KEY_COLS = ["review_id", "product_id", "seller_product_id"]
BAD_KEY_VALUES = {"", "nan", "none", "<na>", "null"}

def normalize_key_value(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lstrip("\ufeff")
    if value.lower() in BAD_KEY_VALUES:
        return ""
    if value.endswith(".0") and value[:-2].isdigit():
        value = value[:-2]
    return value

def make_review_key(review_id, product_id, seller_product_id):
    return tuple(normalize_key_value(v) for v in (review_id, product_id, seller_product_id))

# ==============================================================================
# 1. THIẾT LẬP ĐƯỜNG DẪN HỆ THỐNG
# ==============================================================================
# Tự động tìm thư mục gốc chứa 01_craw_data dù notebook chạy từ thư mục nào
CURRENT_DIR = Path.cwd().resolve()

PROJECT_DIR = next(
    (p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "01_craw_data").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Không tìm thấy thư mục 01_craw_data từ: {CURRENT_DIR}")

# Định nghĩa thư mục sản phẩm và thư mục review theo cấu trúc mới
PRODUCT_DIR = PROJECT_DIR / "01_craw_data" / "product"
REVIEW_DIR = PROJECT_DIR / "01_craw_data" / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# File đầu vào
PRODUCT_DETAIL_FILE = PRODUCT_DIR / "product_detail.csv"
PRODUCT_SELLER_FILE = PRODUCT_DIR / "product_seller.csv"

# File đầu ra review
PRODUCT_REVIEW_FILE = REVIEW_DIR / "product_review.csv"

print(f"Project Dir        : {PROJECT_DIR}")
print(f"Product Detail File: {PRODUCT_DETAIL_FILE} (Tồn tại: {PRODUCT_DETAIL_FILE.exists()})")
print(f"Product Seller File: {PRODUCT_SELLER_FILE} (Tồn tại: {PRODUCT_SELLER_FILE.exists()})")
print(f"Product Review File: {PRODUCT_REVIEW_FILE} (Tồn tại: {PRODUCT_REVIEW_FILE.exists()})")

In [9]:
# ==============================================================================
# 2. CẤU HÌNH THAM SỐ CHẠY
# ==============================================================================
API_URL = "https://tiki.vn/api/v2/reviews"
PAGE_LIMIT = 20            # Số review mỗi trang
SORT_MODE = "id|desc"      # review mới nhất trước để early stop
REQUEST_DELAY = (0.5, 1.2) # Nghỉ giữa các trang VÀ giữa các sản phẩm
SAVE_EVERY_N = 20          # Lưu file sau mỗi N sản phẩm
MAX_RETRIES = 2
EARLY_STOP_EXISTING = True # Dừng phân trang khi gặp review đã lưu
TEST_LIMIT = None          # đặt số (vd 20) để chạy thử; None = chạy toàn bộ

In [ ]:
# ==============================================================================
# 3. CHỌN SẢN PHẨM CẦN CRAWL: review_count (product_detail) > số review đã lưu
# ==============================================================================
if not PRODUCT_DETAIL_FILE.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {PRODUCT_DETAIL_FILE}")

df_detail = pd.read_csv(
    PRODUCT_DETAIL_FILE,
    usecols=["product_id", "review_count", "crawl_date"],
    dtype={"product_id": "string", "crawl_date": "string"},
    low_memory=False,
)
df_detail["product_id"] = df_detail["product_id"].map(normalize_key_value)
df_detail["review_count"] = pd.to_numeric(df_detail["review_count"], errors="coerce").fillna(0).astype(int)
# Chuẩn hóa crawl_date về YYYY-MM (tránh lẫn dạng 2025-08-13 với 2026-10)
df_detail["crawl_date"] = df_detail["crawl_date"].astype(str).str.strip().str[:7]

print("Các tháng có trong product_detail:", sorted(df_detail["crawl_date"].unique().tolist()))

# Lấy review_count của snapshot mới nhất cho mỗi sản phẩm
latest = (
    df_detail.sort_values("crawl_date")
    .drop_duplicates("product_id", keep="last")
    .rename(columns={"review_count": "review_count_current"})
)

# Số review đã lưu theo product_id, khử trùng bằng khóa ghép ba cột đầu
saved_counts = pd.Series(dtype="int64")
if PRODUCT_REVIEW_FILE.exists():
    _rv = pd.read_csv(
        PRODUCT_REVIEW_FILE,
        usecols=REVIEW_KEY_COLS,
        dtype={col: "string" for col in REVIEW_KEY_COLS},
        low_memory=False,
    )
    for col in REVIEW_KEY_COLS:
        _rv[col] = _rv[col].map(normalize_key_value)
    _rv = _rv.drop_duplicates(REVIEW_KEY_COLS)
    saved_counts = _rv.groupby("product_id").size()

latest["review_count_saved"] = latest["product_id"].map(saved_counts).fillna(0).astype(int)
latest["review_count_diff"] = latest["review_count_current"] - latest["review_count_saved"]

df_new_reviews = (
    latest[(latest["review_count_current"] > 0) & (latest["review_count_diff"] > 0)]
    .sort_values("review_count_diff", ascending=False)
    .reset_index(drop=True)
)

print(f"- Tổng sản phẩm trong product_detail      : {len(latest):,}")
print(f"- Sản phẩm cần crawl (còn thiếu review)   : {len(df_new_reviews):,}")
print(f"- Ước tính số review còn thiếu            : {df_new_reviews['review_count_diff'].sum():,}")
print("\nTop 5 sản phẩm còn thiếu review nhiều nhất:")
print(df_new_reviews[["product_id", "review_count_current", "review_count_saved", "review_count_diff"]].head())

In [ ]:
# ==============================================================================
# 4. GHÉP THÔNG TIN SELLER
# ==============================================================================
if not PRODUCT_SELLER_FILE.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {PRODUCT_SELLER_FILE}")

df_seller = pd.read_csv(
    PRODUCT_SELLER_FILE,
    dtype={"product_id": "string", "seller_product_id": "string", "seller_id": "string"},
    low_memory=False,
)

# Làm sạch: chuỗi rỗng / nan / None / <NA> -> thiếu thật sự (NA)
BAD_VALUES = {"", "nan", "none", "<na>", "null"}
for c in ["product_id", "seller_product_id", "seller_id"]:
    s = df_seller[c].astype("string").str.replace("\ufeff", "", regex=False).str.strip()
    df_seller[c] = s.mask(s.str.lower().isin(BAD_VALUES))

df_seller = df_seller.dropna(subset=["product_id", "seller_product_id"])

# Ưu tiên bản ghi is_current_seller == True
if "is_current_seller" in df_seller.columns:
    df_seller["is_curr"] = df_seller["is_current_seller"].astype(str).str.lower().isin(["true", "1"])
    df_seller = df_seller.sort_values(by=["product_id", "is_curr"], ascending=[True, False], kind="stable")

df_seller_unique = df_seller.drop_duplicates(subset=["product_id"], keep="first")[
    ["product_id", "seller_product_id", "seller_id"]
]

df_crawl = pd.merge(df_new_reviews, df_seller_unique, on="product_id", how="left")

# Fallback: thiếu seller_product_id thì dùng product_id; KHÔNG bịa seller_id (để trống)
df_crawl["seller_product_id"] = df_crawl["seller_product_id"].fillna(df_crawl["product_id"])
df_crawl = df_crawl.reset_index(drop=True)

if TEST_LIMIT:
    df_crawl = df_crawl.head(TEST_LIMIT)

print("=" * 60)
print(f"Số sản phẩm sẽ crawl: {len(df_crawl):,}  (thiếu seller_id: {df_crawl['seller_id'].isna().sum():,})")
print("=" * 60)
print(df_crawl[["product_id", "seller_product_id", "seller_id", "review_count_current", "review_count_diff"]].head())

In [ ]:
# ==============================================================================
# 5. KHỞI TẠO SESSION VÀ CÁC HÀM LƯU TRỮ AN TOÀN
# ==============================================================================
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://tiki.vn/"
})

REVIEW_COLUMNS = [
    "review_id", "product_id", "seller_product_id", "seller_id",
    "rating", "title", "content", "review_created_at",
    "purchased", "thank_count", "comment_count", "is_photo", "crawl_time"
]

# Đọc trước các khóa ghép đã có để chống trùng và hỗ trợ Early Stop
existing_review_keys = set()
if PRODUCT_REVIEW_FILE.exists():
    try:
        df_old = pd.read_csv(
            PRODUCT_REVIEW_FILE,
            usecols=REVIEW_KEY_COLS,
            dtype={col: "string" for col in REVIEW_KEY_COLS},
            low_memory=False,
        )
        existing_review_keys = {
            make_review_key(*row)
            for row in df_old[REVIEW_KEY_COLS].itertuples(index=False, name=None)
        }
        print(f"Đã nạp {len(existing_review_keys):,} khóa review ghép để chống ghi trùng và hỗ trợ Early Stop.")
    except Exception as e:
        print(f"Lưu ý khi đọc file review cũ: {e}")

def save_buffer_to_file(buffer_list, output_file):
    """Lưu danh sách review vào file csv an toàn, loại trừ trùng lặp."""
    if not buffer_list:
        return 0

    new_df = pd.DataFrame(buffer_list)
    
    # Đảm bảo đúng thứ tự và đầy đủ cột
    for col in REVIEW_COLUMNS:
        if col not in new_df.columns:
            new_df[col] = None
    new_df = new_df[REVIEW_COLUMNS]

    # Loại bỏ trùng lặp trong buffer và so với file cũ theo khóa ghép
    new_df["_review_key"] = [
        make_review_key(*row)
        for row in new_df[REVIEW_KEY_COLS].itertuples(index=False, name=None)
    ]
    new_df = new_df.drop_duplicates(subset=["_review_key"], keep="first")
    new_df = new_df[~new_df["_review_key"].isin(existing_review_keys)]
    new_keys = new_df["_review_key"].tolist()
    new_df = new_df.drop(columns=["_review_key"])

    if not new_df.empty:
        existing_review_keys.update(new_keys)

        file_exists = output_file.exists() and output_file.stat().st_size > 0
        new_df.to_csv(
            output_file,
            mode="a",
            header=not file_exists,
            index=False,
            encoding="utf-8" if file_exists else "utf-8-sig"
        )
        saved_count = len(new_df)
    else:
        saved_count = 0

    buffer_list.clear()
    return saved_count

In [ ]:
# ==============================================================================
# 6. CRAWL REVIEW (EARLY STOP, DỪNG KHI BỊ CHẶN, CHỈ LƯU SẢN PHẨM CRAWL XONG)
# ==============================================================================
def fetch_reviews_page(params):
    """Trả về (data, status) với status: ok | stop (bị chặn) | error."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = session.get(API_URL, params=params, timeout=20)
            if res.status_code in (401, 403):
                return None, "stop"
            if res.status_code == 429 or res.status_code >= 500:
                time.sleep(10 * attempt)
                continue
            res.raise_for_status()
            return res.json(), "ok"
        except (requests.RequestException, ValueError):
            time.sleep(2 * attempt)
    return None, "error"


review_buffer = []
failed_products = []
total_reviews_saved = 0
crawled_products_count = 0
stopped = False
start_time = time.time()

print(f"Bắt đầu crawl review cho {len(df_crawl)} sản phẩm...")

try:
    for i, row in enumerate(df_crawl.itertuples(index=False), start=1):
        pid = str(row.product_id)
        spid = str(row.seller_product_id)
        seller_id = str(row.seller_id) if pd.notna(row.seller_id) else None

        product_rows, page, ok, hit_existing = [], 1, True, False

        while True:
            params = {
                "limit": PAGE_LIMIT,
                "include": "contribute_info,attribute_vote_summary",
                "sort": SORT_MODE,
                "page": page,
                "spid": spid,
                "product_id": pid,
            }
            if seller_id:
                params["seller_id"] = seller_id

            data, status = fetch_reviews_page(params)

            if status == "stop":
                stopped = True
                ok = False
                print(f"[HTTP 401/403] Bị chặn tại sản phẩm {pid} (trang {page}). Dừng crawl.")
                break
            if status == "error":
                failed_products.append(pid)
                ok = False
                print(f"  [Lỗi] Sản phẩm {pid}, trang {page}: bỏ qua, sẽ thử lại ở lần chạy sau.")
                break

            items = data.get("data") or []
            for r in items:
                rid = r.get("id")
                # Early stop chỉ khi đúng khóa review + sản phẩm + seller product đã lưu
                review_key = make_review_key(rid, pid, spid)
                if EARLY_STOP_EXISTING and review_key in existing_review_keys:
                    hit_existing = True
                    break

                timeline = r.get("timeline") or {}
                creator = r.get("created_by") or {}
                product_rows.append({
                    "review_id": rid,
                    "product_id": pid,
                    "seller_product_id": spid,
                    "seller_id": seller_id,
                    "rating": r.get("rating"),
                    "title": r.get("title"),
                    "content": r.get("content"),
                    "review_created_at": timeline.get("review_created_date"),
                    "purchased": creator.get("purchased"),
                    "thank_count": r.get("thank_count"),
                    "comment_count": r.get("comment_count"),
                    "is_photo": r.get("is_photo"),
                    "crawl_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })

            last_page = (data.get("paging") or {}).get("last_page", 1)
            if hit_existing or not items or page >= last_page:
                break

            page += 1
            time.sleep(random.uniform(*REQUEST_DELAY))

        if stopped:
            break

        # Chỉ nhận review khi cả sản phẩm crawl xong không lỗi
        if ok:
            review_buffer.extend(product_rows)
            crawled_products_count += 1

        if i % SAVE_EVERY_N == 0:
            saved = save_buffer_to_file(review_buffer, PRODUCT_REVIEW_FILE)
            total_reviews_saved += saved
            elapsed = time.time() - start_time
            print(f"[{i}/{len(df_crawl)}] +{saved} review (tổng {total_reviews_saved:,}) | lỗi: {len(failed_products)} | {elapsed:.0f}s")

        time.sleep(random.uniform(*REQUEST_DELAY))   # nghỉ giữa các sản phẩm

finally:
    total_reviews_saved += save_buffer_to_file(review_buffer, PRODUCT_REVIEW_FILE)

print("\n" + "=" * 60)
print("DỪNG SỚM do bị chặn (chạy lại sau để tiếp tục)" if stopped else "HOÀN THÀNH CRAWL REVIEW!")
print(f"- Sản phẩm crawl xong           : {crawled_products_count}/{len(df_crawl)}")
print(f"- Sản phẩm lỗi (chạy lại sau)   : {len(failed_products)}")
print(f"- Review mới đã ghi file        : {total_reviews_saved:,}")
print(f"- File kết quả                  : {PRODUCT_REVIEW_FILE}")
if PRODUCT_REVIEW_FILE.exists():
    print(f"- Dung lượng file review        : {PRODUCT_REVIEW_FILE.stat().st_size / (1024*1024):.2f} MB")
print("=" * 60)
if failed_products:
    print("Một số sản phẩm lỗi:", failed_products[:20])

In [14]:
# ==============================================================================
# 7. KIỂM TRA NHANH KẾT QUẢ CRAWL
# ==============================================================================
if PRODUCT_REVIEW_FILE.exists():
    df_check = pd.read_csv(PRODUCT_REVIEW_FILE, low_memory=False)
    print(f"Tổng số dòng review trong file       : {len(df_check):,}")
    print(f"Số lượng sản phẩm độc nhất có review : {df_check['product_id'].nunique():,}")
    print("\n5 dòng review mới nhất được ghi nhận:")
    cols_to_show = [c for c in ["review_id", "product_id", "rating", "review_created_at", "crawl_time"] if c in df_check.columns]
    print(df_check.tail(5)[cols_to_show])

Tổng số dòng review trong file       : 890,488
Số lượng sản phẩm độc nhất có review : 17,884

5 dòng review mới nhất được ghi nhận:
        review_id  product_id  rating    review_created_at  \
890483   19572382   272062113       5  2023-10-06 12:39:11   
890484   14960348   161538053       5  2022-02-14 21:37:29   
890485   20265646   143694542       5  2026-10-15 18:43:15   
890486   19270171   193207921       5  2023-06-07 17:53:34   
890487   20104638   276449585       2  2025-04-01 11:34:54   

                 crawl_time  
890483  2026-10-20 15:31:48  
890484  2026-10-20 15:31:50  
890485  2026-10-20 15:31:51  
890486  2026-10-20 15:31:52  
890487  2026-10-20 15:31:53  
